In [1]:
import numpy as np
from perf_func import time_stage, test_concurrency
import jax
import jax.numpy as jnp
import numba

1. numpy 

In [2]:
def seq_zadoff_chu(u):
    n = np.arange(63)
    d_u = np.exp(-1j * np.pi * u * n * (n + 1) / 63)
    d_u[31] = 0
    return d_u

def zadoff_chu(u, N_FFT):
    zc = seq_zadoff_chu(u)
    re = np.zeros(N_FFT, complex)
    re[N_FFT // 2 - 31:N_FFT // 2 + 32] = zc
    return np.fft.ifft(np.fft.ifftshift(re))

def tilde_s():
    x = np.zeros(31, dtype=np.uint8)
    x[4] = 1
    for i in range(26):
        x[i + 5] = x[i + 2] ^ x[i]
    return 1 - 2.0 * x

def tilde_c():
    x = np.zeros(31, dtype=np.uint8)
    x[4] = 1
    for i in range(26):
        x[i + 5] = x[i + 3] ^ x[i]
    return 1 - 2.0 * x

def tilde_z():
    x = np.zeros(31, dtype=np.uint8)
    x[4] = 1
    for i in range(26):
        x[i + 5] = x[i + 4] ^ x[i + 2] ^ x[i + 1] ^ x[i]
    return 1 - 2.0 * x

def m_01(N_id_1):
    q_prime = N_id_1 // 30
    q = (N_id_1 + q_prime * (q_prime + 1) / 2) // 30
    m_prime = N_id_1 + q * (q + 1) / 2
    m_0 = int(m_prime % 31)
    m_1 = int((m_0 + m_prime // 31 + 1) % 31)
    return (m_0, m_1)

def m_sequence(N_id_1, N_id_2, F, N_FFT):
    m_0, m_1 = m_01(N_id_1)
    ts, tc, tz = tilde_s(), tilde_c(), tilde_z()
    c_0 = np.roll(tc, -N_id_2)
    c_1 = np.roll(tc, -N_id_2 - 3)
    s_0, s_1 = np.roll(ts, -m_0), np.roll(ts, -m_1)
    z_10, z_11 = np.roll(tz, -(m_0 % 8)), np.roll(tz, -(m_1 % 8))
    d = np.zeros(62)
    if F == 0:
        d[0::2] = s_0 * c_0
        d[1::2] = s_1 * c_1 * z_10
    elif F == 1:
        d[0::2] = s_1 * c_0
        d[1::2] = s_0 * c_1 * z_11
    re = np.zeros(N_FFT)
    re[N_FFT // 2 - 31:N_FFT // 2] = d[:31]
    re[N_FFT // 2 + 1:N_FFT // 2 + 32] = d[31:]
    return np.fft.ifft(np.fft.ifftshift(re))


In [3]:
class PSSDetection:

    def __init__(self, params, peak_ratio=5.0):
        self.N_FFT = params.N_FFT
        self.N_CP = params.N_CP
        self.peak_ratio = peak_ratio

        roots = [25, 29, 34]
        self.pss_refs = {n: zadoff_chu(roots[n], params.N_FFT) for n in range(3)}
        self.pss_norms = {n: np.linalg.norm(self.pss_refs[n]) for n in range(3)}

    def __call__(self, chunk):
        chunk.rx_norms = self._sliding_window_energy(chunk.data)
        rx_fft = np.fft.fft(chunk.data) 

        best_ratio, best_pos, best_nid2 = 0.0, None, None

        for n in range(3):
            norm_corr = self._correlate(rx_fft, chunk.rx_norms, self.pss_refs[n], self.pss_norms[n])
            pos, ratio = self._find_peak(norm_corr)

            if ratio > best_ratio:
                best_ratio = ratio
                best_pos = pos
                best_nid2 = n

        if best_ratio > self.peak_ratio and best_pos >= self.N_CP + self.N_FFT:
            chunk.pss_detected = True
            chunk.pss_local_index = best_pos
            chunk.N_id_2 = best_nid2

        return chunk
    
    def _sliding_window_energy(self, rx):
        """Energy of each N-sample window using prefix sum."""  
        power = np.abs(rx) ** 2
        cs = np.concatenate(([0], np.cumsum(power)))
        energy = cs[self.N_FFT:] - cs[:len(rx) - self.N_FFT + 1]
        return np.sqrt(energy)
    
    def _correlate(self, rx_fft, rx_norms, pss_ref, pss_norm):
        """Normalized FFT cross-correlation against one template."""
        L = len(rx_fft)
        pss_padded = np.zeros(L, dtype=complex)
        pss_padded[:self.N_FFT] = pss_ref

        corr_abs = np.abs(np.fft.ifft(rx_fft * np.conj(np.fft.fft(pss_padded))))[:L - self.N_FFT + 1]

        valid = rx_norms > 0
        norm_corr = np.zeros_like(corr_abs)
        norm_corr[valid] = corr_abs[valid] / (rx_norms[valid] * pss_norm)
        return norm_corr
    
    def _find_peak(self, norm_corr):
        """Peak-to-median ratio. High ratio = real PSS, low = noise."""
        valid = norm_corr > 0
        peak_pos = np.argmax(norm_corr)
        peak_val = norm_corr[peak_pos]
        median_val = np.median(norm_corr[valid])
        ratio = peak_val / median_val if median_val > 0 else 0.0
        return peak_pos, ratio

In [4]:
class SSSDetection:

    def __init__(self, params, peak_ratio=5.0):
        self.N_FFT = params.N_FFT
        self.N_CP = params.N_CP
        self.Fs = params.Fs
        self.peak_ratio = peak_ratio

        roots = [25, 29, 34]
        self.pss_refs = {n: zadoff_chu(roots[n], params.N_FFT) for n in range(3)}

        # pre-compute SSS references: 336 candidates per N_id_2
        self.sss_refs = {}
        for nid2 in range(3):
            self.sss_refs[nid2] = {}
            for N_id_1 in range(168):
                for F in range(2):
                    idx = N_id_1 + F * 168
                    sig = m_sequence(N_id_1, nid2, F, params.N_FFT)
                    self.sss_refs[nid2][idx] = {
                        'N_id_1': N_id_1,
                        'F': F,
                        'sig': sig,
                        'norm': np.linalg.norm(sig)}

    def __call__(self, chunk):
        if not chunk.pss_detected:
            return chunk
        sss_start = chunk.pss_local_index - self.N_CP - self.N_FFT
        sss_rx = chunk.data[sss_start:sss_start + self.N_FFT]
        sss_rx_norm = chunk.rx_norms[sss_start]

        best_idx, ratio = self._search_candidates(sss_rx, sss_rx_norm, chunk.N_id_2)
        
        if ratio > self.peak_ratio:
            ref = self.sss_refs[chunk.N_id_2][best_idx]
            chunk.sss_detected = True
            chunk.N_id_1 = ref['N_id_1']
            chunk.F = ref['F']
            chunk.f_d = self._estimate_freq_offset(chunk)

        return chunk

    def _search_candidates(self, rx, rx_norm, N_id_2):
        """Correlate against all 336 candidates, return best index and ratio."""
        refs = self.sss_refs[N_id_2]
        ref_keys = list(refs.keys())

        corr_vals = np.empty(len(refs))
        for i, idx in enumerate(ref_keys):
            info = refs[idx]
            corr_vals[i] = np.abs(np.sum(rx * np.conj(info['sig']))) / \
                           (rx_norm * info['norm'])

        best_i = np.argmax(corr_vals)
        peak_val = corr_vals[best_i]
        median_val = np.median(corr_vals)
        ratio = peak_val / median_val if median_val > 0 else 0.0

        return ref_keys[best_i], ratio

    def _estimate_freq_offset(self, chunk):
        pss_rx = chunk.data[chunk.pss_local_index:chunk.pss_local_index + self.N_FFT]
        pss_demod = pss_rx * np.conj(self.pss_refs[chunk.N_id_2])

        pl = np.sum(pss_demod[:self.N_FFT // 2])
        pu = np.sum(pss_demod[self.N_FFT // 2:])

        return np.angle(pu * np.conj(pl)) / (2 * np.pi * self.N_FFT // 2) * self.Fs

In [5]:
class PSSChunk:
    """simple data structure to test accuracy and no effect on speed test"""
    def __init__(self, data, tag):
        self.data = data
        self.tag = tag

        # PSS results
        self.rx_norms = None 
        self.pss_detected = False
        self.pss_local_index = None
        self.N_id_2 = None

        # SSS results
        self.sss_detected = False
        self.N_id_1 = None
        self.F = None
        self.f_d = None  

accuracy + time cost for numpy

In [6]:
from data.lte_system_info import LTEParams

In [7]:
rxf = np.load('data/rx_preprocessed.npy')
Fs = float(np.load('data/Fs.npy'))
params = LTEParams(Fs=Fs)

In [8]:
N_overlap = params.N_CP + 2 * params.N_FFT
N_subframe = params.N_subframe
stride = N_subframe - N_overlap

pss = PSSDetection(params, peak_ratio=5.0)

pos = 0
chunk_id = 0
pss_count = 0
cells = []

while pos + N_subframe <= len(rxf):

    data = rxf[pos:pos+N_subframe]
    chunk = PSSChunk(data, chunk_id)
    chunk = pss(chunk)

    if chunk.pss_detected:
        pss_count += 1
        global_pss = pos + chunk.pss_local_index
        print(f"Chunk {chunk_id}: PSS detected | "
              f"N_id_2={chunk.N_id_2}, "
              f"local={chunk.pss_local_index}, global={global_pss}")
        cells.append(chunk)    

    pos += stride
    chunk_id += 1

print(f"\nScanned {chunk_id} chunks over {len(rxf)} samples, "
      f"{pss_count} PSS detected")

Chunk 0: PSS detected | N_id_2=2, local=12309, global=12309
Chunk 2: PSS detected | N_id_2=2, local=9563, global=36043
Chunk 6: PSS detected | N_id_2=2, local=9667, global=89107
Chunk 8: PSS detected | N_id_2=2, local=6922, global=112842
Chunk 12: PSS detected | N_id_2=2, local=7027, global=165907
Chunk 13: PSS detected | N_id_2=0, local=11192, global=183312
Chunk 14: PSS detected | N_id_2=2, local=4283, global=189643
Chunk 18: PSS detected | N_id_2=2, local=4390, global=242710
Chunk 19: PSS detected | N_id_2=2, local=14327, global=265887
Chunk 20: PSS detected | N_id_2=2, local=1642, global=266442
Chunk 22: PSS detected | N_id_2=2, local=14330, global=305610
Chunk 24: PSS detected | N_id_2=2, local=1746, global=319506
Chunk 25: PSS detected | N_id_2=2, local=12243, global=343243
Chunk 26: PSS detected | N_id_2=1, local=2587, global=346827
Chunk 29: PSS detected | N_id_2=2, local=12349, global=396309
Chunk 31: PSS detected | N_id_2=2, local=9602, global=420042
Chunk 35: PSS detected | 

In [9]:
sss = SSSDetection(params, peak_ratio=8.0)

sss_cells = []
for chunk in cells:
    chunk = sss(chunk)
    if chunk.sss_detected:
        global_pss = chunk.tag * stride + chunk.pss_local_index
        PCI = 3 * chunk.N_id_1 + chunk.N_id_2
        print(f"Chunk {chunk.tag}: PCI={PCI} | "
              f"N_id_1={chunk.N_id_1}, N_id_2={chunk.N_id_2}, F={chunk.F}, "
              f"local={chunk.pss_local_index}, global={global_pss} | "
              f"f_d={chunk.f_d:.1f}")
        sss_cells.append(chunk)

print(f"\n{len(sss_cells)} SSS detected out of {len(cells)} PSS chunks")

if sss_cells:
    first_global = sss_cells[0].tag * stride + sss_cells[0].pss_local_index
    expected = (len(rxf) - first_global) // params.N_half_frame + 1
    print(f"Expected {expected} cell found ({params.N_half_frame} spacing from {first_global})")

Chunk 2: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=9563, global=36043 | f_d=1126.9
Chunk 8: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=6922, global=112842 | f_d=1128.4
Chunk 14: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=4283, global=189643 | f_d=1164.2
Chunk 20: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=1642, global=266442 | f_d=1204.1
Chunk 25: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=12243, global=343243 | f_d=1180.9
Chunk 31: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=9602, global=420042 | f_d=1107.1
Chunk 37: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=6962, global=496842 | f_d=1179.5
Chunk 43: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=4323, global=573643 | f_d=1142.0
Chunk 49: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=1681, global=650441 | f_d=1269.2
Chunk 54: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=12282, global=727242 | f_d=1165.1
Chunk 60: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=9642, global=804042 | f_d=1122.2
Chunk 66: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=70

In [10]:
# 1. unit time
pss_time = time_stage(pss, cells[0])
print(f"PSS unit time: {pss_time:.2f} ms\n")
 
# 2. concurrency
results = test_concurrency(pss, cells)
 
print(f"PSS concurrency test: {len(cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000    
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

PSS unit time: 1.54 ms

PSS concurrency test: 566 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        1.67     1.00x
       2     2        0.93     1.80x
       4     4        0.58     2.90x
       6     6        0.59     2.83x
      10    10        0.71     2.35x


In [11]:
# 1. unit time
sss_time = time_stage(sss, sss_cells[0])
print(f"SSS unit time: {sss_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(sss, sss_cells)

print(f"SSS concurrency test: {len(sss_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(sss_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

SSS unit time: 1.77 ms

SSS concurrency test: 200 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        1.83     1.00x
       2     2        2.82     0.65x
       4     4        4.14     0.44x
       6     6        4.49     0.41x
      10    10        4.65     0.39x


2. Jax

In [12]:
class PSSDetectionJAX:

    def __init__(self, params, peak_ratio=5.0):
        self.N_FFT = params.N_FFT
        self.N_CP = params.N_CP
        self.peak_ratio = peak_ratio

        roots = [25, 29, 34]
        self.pss_refs = {n: zadoff_chu(roots[n], params.N_FFT) for n in range(3)}
        self.pss_norms = {n: np.linalg.norm(self.pss_refs[n]) for n in range(3)}

        # JAX precompute: stack refs, precompute conj(fft(padded_ref))
        self._jax_pss_norms = jnp.array([self.pss_norms[n] for n in range(3)])
        self._jax_pss_refs_fft = jnp.stack([
            jnp.conj(jnp.fft.fft(
                jnp.zeros(params.N_subframe, dtype=complex).at[:params.N_FFT].set(self.pss_refs[n])
            ))
            for n in range(3)
        ])

        # warmup JIT
        _dummy = jnp.zeros(params.N_subframe, dtype=complex)
        PSSDetectionJAX._jax_detect(_dummy, self._jax_pss_refs_fft, self._jax_pss_norms, self.N_FFT)

    def __call__(self, chunk):
        rx_norms, best_nid2, best_pos, best_ratio = PSSDetectionJAX._jax_detect(
            jnp.array(chunk.data),
            self._jax_pss_refs_fft, self._jax_pss_norms, self.N_FFT)

        chunk.rx_norms = np.array(rx_norms)

        if float(best_ratio) > self.peak_ratio and int(best_pos) >= self.N_CP + self.N_FFT:
            chunk.pss_detected = True
            chunk.pss_local_index = int(best_pos)
            chunk.N_id_2 = int(best_nid2)

        return chunk

    @staticmethod
    @jax.jit(static_argnums=(3,))
    def _jax_detect(data, pss_refs_fft, pss_norms, N_FFT):
        rx_norms = PSSDetectionJAX._sliding_window_energy(data, N_FFT)
        rx_fft = jnp.fft.fft(data)
        norm_corr = PSSDetectionJAX._correlate(rx_fft, rx_norms, pss_refs_fft, pss_norms, N_FFT)
        best_nid2, best_pos, best_ratio = PSSDetectionJAX._find_peak(norm_corr)
        return rx_norms, best_nid2, best_pos, best_ratio

    @staticmethod
    def _sliding_window_energy(data, N_FFT):
        L = data.shape[0]
        power = jnp.abs(data) ** 2
        cs = jnp.concatenate([jnp.array([0.0]), jnp.cumsum(power)])
        return jnp.sqrt(cs[N_FFT:] - cs[:L - N_FFT + 1])

    @staticmethod
    def _correlate(rx_fft, rx_norms, pss_refs_fft, pss_norms, N_FFT):
        L = rx_fft.shape[0]
        corr_abs = jnp.abs(jnp.fft.ifft(rx_fft[None, :] * pss_refs_fft))[:, :L - N_FFT + 1]
        denom = rx_norms[None, :] * pss_norms[:, None]
        safe_denom = jnp.where(denom > 0, denom, 1.0)
        return jnp.where(denom > 0, corr_abs / safe_denom, 0.0)

    @staticmethod
    def _find_peak(norm_corr):
        peak_pos = jnp.argmax(norm_corr, axis=1)
        peak_vals = norm_corr[jnp.arange(3), peak_pos]
        medians = jnp.median(norm_corr, axis=1)
        safe_med = jnp.where(medians > 0, medians, 1.0)
        ratios = jnp.where(medians > 0, peak_vals / safe_med, 0.0)
        best_nid2 = jnp.argmax(ratios)
        return best_nid2, peak_pos[best_nid2], ratios[best_nid2]

In [13]:
class SSSDetectionJAX:

    def __init__(self, params, peak_ratio=5.0):
        self.N_FFT = params.N_FFT
        self.N_CP = params.N_CP
        self.Fs = params.Fs
        self.peak_ratio = peak_ratio

        roots = [25, 29, 34]
        self.pss_refs = {n: zadoff_chu(roots[n], params.N_FFT) for n in range(3)}

        # pre-compute SSS references
        self.sss_refs = {}
        for nid2 in range(3):
            self.sss_refs[nid2] = {}
            for N_id_1 in range(168):
                for F in range(2):
                    idx = N_id_1 + F * 168
                    sig = m_sequence(N_id_1, nid2, F, params.N_FFT)
                    self.sss_refs[nid2][idx] = {
                        'N_id_1': N_id_1,
                        'F': F,
                        'sig': sig,
                        'norm': np.linalg.norm(sig)}

        # JAX precompute: stack all 336 templates into matrices
        self._jax_refs = {}
        for nid2 in range(3):
            refs = self.sss_refs[nid2]
            ref_keys = list(refs.keys())
            sig_matrix = jnp.stack([refs[k]['sig'] for k in ref_keys])
            norm_array = jnp.array([refs[k]['norm'] for k in ref_keys])
            self._jax_refs[nid2] = {
                'keys': ref_keys,
                'sig_matrix': sig_matrix,
                'norms': norm_array
            }

        # warmup JIT
        _dummy = jnp.zeros(params.N_FFT, dtype=complex)
        _dummy_norms = jnp.ones(336)
        _dummy_matrix = self._jax_refs[0]['sig_matrix']
        SSSDetectionJAX._jax_search(_dummy, 1.0, _dummy_matrix, _dummy_norms)

    def __call__(self, chunk):
        if not chunk.pss_detected:
            return chunk

        sss_start = chunk.pss_local_index - self.N_CP - self.N_FFT
        sss_rx = chunk.data[sss_start:sss_start + self.N_FFT]
        sss_rx_norm = chunk.rx_norms[sss_start]

        refs_j = self._jax_refs[chunk.N_id_2]
        best_i, ratio = SSSDetectionJAX._jax_search(
            jnp.array(sss_rx), sss_rx_norm,
            refs_j['sig_matrix'], refs_j['norms'])

        if float(ratio) > self.peak_ratio:
            best_idx = refs_j['keys'][int(best_i)]
            ref = self.sss_refs[chunk.N_id_2][best_idx]
            chunk.sss_detected = True
            chunk.N_id_1 = ref['N_id_1']
            chunk.F = ref['F']
            chunk.f_d = self._estimate_freq_offset(chunk)

        return chunk

    @staticmethod
    @jax.jit
    def _jax_search(sss_rx, rx_norm, sig_matrix, norms):
        corr_vals = jnp.abs(sig_matrix.conj() @ sss_rx) / (rx_norm * norms)
        best_i = jnp.argmax(corr_vals)
        peak_val = corr_vals[best_i]
        median_val = jnp.median(corr_vals)
        ratio = peak_val / median_val
        return best_i, ratio

    def _estimate_freq_offset(self, chunk):
        N = self.N_FFT
        pss_rx = chunk.data[chunk.pss_local_index:chunk.pss_local_index + N]
        pss_demod = pss_rx * np.conj(self.pss_refs[chunk.N_id_2])
        pl = np.sum(pss_demod[:N // 2])
        pu = np.sum(pss_demod[N // 2:])
        return np.angle(pu * np.conj(pl)) / (2 * np.pi * N // 2) * self.Fs

accuracy + time cost for jax

In [14]:
N_overlap = params.N_CP + 2 * params.N_FFT
N_subframe = params.N_subframe
stride = N_subframe - N_overlap

jax_pss = PSSDetectionJAX(params, peak_ratio=5.0)

pos = 0
chunk_id = 0
pss_count = 0
jax_cells = []

while pos + N_subframe <= len(rxf):

    data = rxf[pos:pos+N_subframe]
    chunk = PSSChunk(data, chunk_id)
    chunk = jax_pss(chunk)

    if chunk.pss_detected:
        pss_count += 1
        global_pss = pos + chunk.pss_local_index
        print(f"Chunk {chunk_id}: PSS detected | "
              f"N_id_2={chunk.N_id_2}, "
              f"local={chunk.pss_local_index}, global={global_pss}")
        jax_cells.append(chunk)    

    pos += stride
    chunk_id += 1

print(f"\nScanned {chunk_id} chunks over {len(rxf)} samples, "
      f"{pss_count} PSS detected")

Chunk 0: PSS detected | N_id_2=2, local=12309, global=12309
Chunk 2: PSS detected | N_id_2=2, local=9563, global=36043
Chunk 6: PSS detected | N_id_2=2, local=9667, global=89107
Chunk 8: PSS detected | N_id_2=2, local=6922, global=112842
Chunk 12: PSS detected | N_id_2=2, local=7027, global=165907
Chunk 13: PSS detected | N_id_2=0, local=11192, global=183312
Chunk 14: PSS detected | N_id_2=2, local=4283, global=189643
Chunk 18: PSS detected | N_id_2=2, local=4390, global=242710
Chunk 19: PSS detected | N_id_2=2, local=14327, global=265887
Chunk 20: PSS detected | N_id_2=2, local=1642, global=266442
Chunk 22: PSS detected | N_id_2=2, local=14330, global=305610
Chunk 24: PSS detected | N_id_2=2, local=1746, global=319506
Chunk 25: PSS detected | N_id_2=2, local=12243, global=343243
Chunk 26: PSS detected | N_id_2=1, local=2587, global=346827
Chunk 29: PSS detected | N_id_2=2, local=12349, global=396309
Chunk 31: PSS detected | N_id_2=2, local=9602, global=420042
Chunk 35: PSS detected | 

In [15]:
jax_sss = SSSDetectionJAX(params, peak_ratio=8.0)

jax_sss_cells = []
for chunk in jax_cells:
    chunk = jax_sss(chunk)
    if chunk.sss_detected:
        global_pss = pos + chunk.pss_local_index
        PCI = 3*chunk.N_id_1 + chunk.N_id_2
        print(f"Chunk {chunk_id}: PCI={PCI} | "
              f"N_id_1={chunk.N_id_1}, N_id_2={chunk.N_id_2}, F={chunk.F}, "
              f"local={chunk.pss_local_index}, global={global_pss} | "
              f"f_d={chunk.f_d}")
        jax_sss_cells.append(chunk)

print(f"{len(jax_sss_cells)} SSS detected out of {len(jax_cells)} PSS chunks")

if jax_sss_cells:
    first_global = jax_sss_cells[0].tag * stride + jax_sss_cells[0].pss_local_index
    expected = (len(rxf) - first_global) // params.N_half_frame + 1
    print(f"Expected {expected} cell found ({params.N_half_frame} spacing from {first_global})")

Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=9563, global=15354723 | f_d=1126.9277884717471
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=6922, global=15352082 | f_d=1128.3884871750954
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=4283, global=15349443 | f_d=1164.1963128217092
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=1642, global=15346802 | f_d=1204.0518627997699
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=12243, global=15357403 | f_d=1180.93824589012
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=9602, global=15354762 | f_d=1107.0686391594347
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=6962, global=15352122 | f_d=1179.5097590630232
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=4323, global=15349483 | f_d=1141.9835229524856
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=1681, global=15346841 | f_d=1269.175516520341
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=12282, global=15357442 | f_d=

In [16]:
# 1. unit time
pss_time = time_stage(jax_pss, jax_cells[0])
print(f"PSS unit time: {pss_time:.2f} ms\n")
 
# 2. concurrency
results = test_concurrency(jax_pss, jax_cells)
 
print(f"PSS concurrency test: {len(jax_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(jax_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

PSS unit time: 7.01 ms

PSS concurrency test: 566 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        7.17     1.00x
       2     2        3.54     2.03x
       4     4        1.98     3.61x
       6     6        1.47     4.89x
      10    10        0.95     7.52x


In [17]:
# 1. unit time
sss_time = time_stage(jax_sss, jax_sss_cells[0])
print(f"JAX SSS unit time: {sss_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(jax_sss, jax_sss_cells)

print(f"JAX SSS concurrency test: {len(jax_sss_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(jax_sss_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

JAX SSS unit time: 0.41 ms

JAX SSS concurrency test: 200 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        0.31     1.00x
       2     2        0.26     1.19x
       4     4        0.25     1.22x
       6     6        0.30     1.05x
      10    10        0.29     1.07x


3. numba

In [18]:
@numba.jit(nopython=True, nogil=True)
def _normalize_and_find_peak(corr, rx_norms, pss_norms):
    """Normalize correlations and find best peak across all 3 PSS roots.

    corr:      (3, L) complex — raw ifft output, already truncated
    rx_norms:  (L,)   float  — sliding window energy
    pss_norms: (3,)   float  — norm of each PSS ref

    Entire function runs with GIL released.
    """
    n_refs = corr.shape[0]
    L = corr.shape[1]

    best_ratio = 0.0
    best_pos = 0
    best_nid2 = 0

    for n in range(n_refs):
        pss_norm = pss_norms[n]

        peak_val = 0.0
        peak_pos_n = 0
        count = 0

        norm_vals = np.empty(L)
        for i in range(L):
            abs_val = abs(corr[n, i])
            if rx_norms[i] > 0:
                norm_vals[i] = abs_val / (rx_norms[i] * pss_norm)
            else:
                norm_vals[i] = 0.0

            if norm_vals[i] > peak_val:
                peak_val = norm_vals[i]
                peak_pos_n = i

            if norm_vals[i] > 0:
                count += 1

        if count > 0:
            pos_vals = np.empty(count)
            j = 0
            for i in range(L):
                if norm_vals[i] > 0:
                    pos_vals[j] = norm_vals[i]
                    j += 1
            pos_vals.sort()

            if count % 2 == 1:
                median_val = pos_vals[count // 2]
            else:
                median_val = (pos_vals[count // 2 - 1] + pos_vals[count // 2]) / 2.0

            ratio = peak_val / median_val if median_val > 0 else 0.0
        else:
            ratio = 0.0

        if ratio > best_ratio:
            best_ratio = ratio
            best_pos = peak_pos_n
            best_nid2 = n

    return best_nid2, best_pos, best_ratio


@numba.jit(nopython=True, nogil=True)
def _sss_search_numba(sss_rx, rx_norm, sig_matrix, norms):
    """Correlate against all 336 SSS candidates. GIL released.

    sss_rx:     (N_FFT,) complex — received SSS symbol
    rx_norm:    float            — energy norm at SSS position
    sig_matrix: (336, N_FFT) complex — precomputed reference signals
    norms:      (336,) float     — norm of each reference

    Returns (best_index, peak_to_median_ratio).
    """
    n_candidates = sig_matrix.shape[0]
    N = sig_matrix.shape[1]

    corr_vals = np.empty(n_candidates)
    for i in range(n_candidates):
        re = 0.0
        im = 0.0
        for j in range(N):
            s = sig_matrix[i, j]
            r = sss_rx[j]
            re += s.real * r.real + s.imag * r.imag
            im += s.real * r.imag - s.imag * r.real
        corr_vals[i] = np.sqrt(re * re + im * im) / (rx_norm * norms[i])

    # find peak
    best_i = 0
    peak_val = corr_vals[0]
    for i in range(1, n_candidates):
        if corr_vals[i] > peak_val:
            peak_val = corr_vals[i]
            best_i = i

    # median
    sorted_vals = corr_vals.copy()
    sorted_vals.sort()
    n = n_candidates
    if n % 2 == 1:
        median_val = sorted_vals[n // 2]
    else:
        median_val = (sorted_vals[n // 2 - 1] + sorted_vals[n // 2]) / 2.0

    ratio = peak_val / median_val if median_val > 0 else 0.0
    return best_i, ratio

In [19]:
class PSSDetectionNumba:

    def __init__(self, params, peak_ratio=5.0):
        self.N_FFT = params.N_FFT
        self.N_CP = params.N_CP
        self.peak_ratio = peak_ratio

        roots = [25, 29, 34]
        self._pss_refs = [zadoff_chu(roots[n], params.N_FFT) for n in range(3)]
        self.pss_norms = np.array([np.linalg.norm(r) for r in self._pss_refs])

        # cache refs for current data length
        self._cached_L = None
        self.refs_fft_conj = None
        self._update_refs(params.N_subframe)

        # warmup numba
        dummy_corr = np.zeros((3, 10), dtype=complex)
        dummy_norms = np.ones(10)
        _normalize_and_find_peak(dummy_corr, dummy_norms, self.pss_norms)

    def _update_refs(self, L):
        self.refs_fft_conj = np.stack([
            np.conj(np.fft.fft(np.pad(self._pss_refs[n], (0, L - self.N_FFT))))
            for n in range(3)
        ])
        self._cached_L = L

    def __call__(self, chunk):
        L = len(chunk.data)
        if L != self._cached_L:
            self._update_refs(L)

        chunk.rx_norms = self._sliding_window_energy(chunk.data)
        rx_fft = np.fft.fft(chunk.data)
        corr = np.fft.ifft(rx_fft[None, :] * self.refs_fft_conj, axis=1)
        L_corr = len(chunk.rx_norms)
        corr = corr[:, :L_corr]

        best_nid2, best_pos, best_ratio = _normalize_and_find_peak(
            corr, chunk.rx_norms, self.pss_norms)

        if best_ratio > self.peak_ratio and best_pos >= self.N_CP + self.N_FFT:
            chunk.pss_detected = True
            chunk.pss_local_index = best_pos
            chunk.N_id_2 = best_nid2

        return chunk

    def _sliding_window_energy(self, rx):
        power = np.abs(rx) ** 2
        cs = np.concatenate(([0], np.cumsum(power)))
        energy = cs[self.N_FFT:] - cs[:len(rx) - self.N_FFT + 1]
        return np.sqrt(energy)

In [20]:
class SSSDetectionNumba:
    """Hybrid SSS: precomputed refs + Numba nogil for candidate search.

    The 336-candidate correlation loop is the hot path.
    No FFT needed — just complex dot products, perfect for Numba.
    Frequency offset estimation stays in NumPy (small, runs once).
    """

    def __init__(self, params, peak_ratio=5.0):
        self.N_FFT = params.N_FFT
        self.N_CP = params.N_CP
        self.Fs = params.Fs
        self.peak_ratio = peak_ratio

        roots = [25, 29, 34]
        self.pss_refs = {n: zadoff_chu(roots[n], params.N_FFT) for n in range(3)}

        # pre-compute SSS references
        self.sss_refs = {}
        for nid2 in range(3):
            self.sss_refs[nid2] = {}
            for N_id_1 in range(168):
                for F in range(2):
                    idx = N_id_1 + F * 168
                    sig = m_sequence(N_id_1, nid2, F, params.N_FFT)
                    self.sss_refs[nid2][idx] = {
                        'N_id_1': N_id_1,
                        'F': F,
                        'sig': sig,
                        'norm': np.linalg.norm(sig)}

        # stack into numpy arrays for numba
        self._np_refs = {}
        for nid2 in range(3):
            refs = self.sss_refs[nid2]
            ref_keys = list(refs.keys())
            sig_matrix = np.stack([refs[k]['sig'] for k in ref_keys])
            norm_array = np.array([refs[k]['norm'] for k in ref_keys])
            self._np_refs[nid2] = {
                'keys': ref_keys,
                'sig_matrix': sig_matrix,
                'norms': norm_array
            }

        # warmup numba
        _dummy = np.zeros(params.N_FFT, dtype=complex)
        _sss_search_numba(_dummy, 1.0,
                          self._np_refs[0]['sig_matrix'],
                          self._np_refs[0]['norms'])

    def __call__(self, chunk):
        if not chunk.pss_detected:
            return chunk

        sss_start = chunk.pss_local_index - self.N_CP - self.N_FFT
        sss_rx = chunk.data[sss_start:sss_start + self.N_FFT]
        sss_rx_norm = chunk.rx_norms[sss_start]

        refs = self._np_refs[chunk.N_id_2]
        best_i, ratio = _sss_search_numba(
            sss_rx, sss_rx_norm,
            refs['sig_matrix'], refs['norms'])

        if ratio > self.peak_ratio:
            best_idx = refs['keys'][int(best_i)]
            ref = self.sss_refs[chunk.N_id_2][best_idx]
            chunk.sss_detected = True
            chunk.N_id_1 = ref['N_id_1']
            chunk.F = ref['F']
            chunk.f_d = self._estimate_freq_offset(chunk)

        return chunk

    def _estimate_freq_offset(self, chunk):
        N = self.N_FFT
        pss_rx = chunk.data[chunk.pss_local_index:chunk.pss_local_index + N]
        pss_demod = pss_rx * np.conj(self.pss_refs[chunk.N_id_2])
        pl = np.sum(pss_demod[:N // 2])
        pu = np.sum(pss_demod[N // 2:])
        return np.angle(pu * np.conj(pl)) / (2 * np.pi * N // 2) * self.Fs

In [21]:
N_overlap = params.N_CP + 2 * params.N_FFT
N_subframe = params.N_subframe
stride = N_subframe - N_overlap

numba_pss = PSSDetectionNumba(params, peak_ratio=5.0)

pos = 0
chunk_id = 0
pss_count = 0
numba_cells = []

while pos + N_subframe <= len(rxf):

    data = rxf[pos:pos+N_subframe]
    chunk = PSSChunk(data, chunk_id)
    chunk = numba_pss(chunk)

    if chunk.pss_detected:
        pss_count += 1
        global_pss = pos + chunk.pss_local_index
        print(f"Chunk {chunk_id}: PSS detected | "
              f"N_id_2={chunk.N_id_2}, "
              f"local={chunk.pss_local_index}, global={global_pss}")
        numba_cells.append(chunk)    

    pos += stride
    chunk_id += 1

print(f"\nScanned {chunk_id} chunks over {len(rxf)} samples, "
      f"{pss_count} PSS detected")

Chunk 0: PSS detected | N_id_2=2, local=12309, global=12309
Chunk 2: PSS detected | N_id_2=2, local=9563, global=36043
Chunk 6: PSS detected | N_id_2=2, local=9667, global=89107
Chunk 8: PSS detected | N_id_2=2, local=6922, global=112842
Chunk 12: PSS detected | N_id_2=2, local=7027, global=165907
Chunk 13: PSS detected | N_id_2=0, local=11192, global=183312
Chunk 14: PSS detected | N_id_2=2, local=4283, global=189643
Chunk 18: PSS detected | N_id_2=2, local=4390, global=242710
Chunk 19: PSS detected | N_id_2=2, local=14327, global=265887
Chunk 20: PSS detected | N_id_2=2, local=1642, global=266442
Chunk 22: PSS detected | N_id_2=2, local=14330, global=305610
Chunk 24: PSS detected | N_id_2=2, local=1746, global=319506
Chunk 25: PSS detected | N_id_2=2, local=12243, global=343243
Chunk 26: PSS detected | N_id_2=1, local=2587, global=346827
Chunk 29: PSS detected | N_id_2=2, local=12349, global=396309
Chunk 31: PSS detected | N_id_2=2, local=9602, global=420042
Chunk 35: PSS detected | 

In [22]:
numba_sss = SSSDetectionNumba(params, peak_ratio=8.0)

numba_sss_cells = []
for chunk in numba_cells:
    chunk = numba_sss(chunk)
    if chunk.sss_detected:
        global_pss = pos + chunk.pss_local_index
        PCI = 3*chunk.N_id_1 + chunk.N_id_2
        print(f"Chunk {chunk_id}: PCI={PCI} | "
              f"N_id_1={chunk.N_id_1}, N_id_2={chunk.N_id_2}, F={chunk.F}, "
              f"local={chunk.pss_local_index}, global={global_pss} | "
              f"f_d={chunk.f_d}")
        numba_sss_cells.append(chunk)

print(f"{len(numba_sss_cells)} SSS detected out of {len(numba_cells)} PSS chunks")

pss_spacing = 5 * N_subframe
first_pss = numba_cells[0].tag * stride + numba_cells[0].pss_local_index
expected = (len(rxf) - first_pss) // pss_spacing + 1
print(f"Expected {expected} cell found ({pss_spacing} spacing from {first_pss})")

Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=9563, global=15354723 | f_d=1126.9277884717471
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=6922, global=15352082 | f_d=1128.3884871750954
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=4283, global=15349443 | f_d=1164.1963128217092
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=1642, global=15346802 | f_d=1204.0518627997699
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=12243, global=15357403 | f_d=1180.93824589012
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=9602, global=15354762 | f_d=1107.0686391594347
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=6962, global=15352122 | f_d=1179.5097590630232
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=4323, global=15349483 | f_d=1141.9835229524856
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=1681, global=15346841 | f_d=1269.175516520341
Chunk 1159: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=12282, global=15357442 | f_d=

accuracy + time cost for numba

In [23]:
# 1. unit time
pss_time = time_stage(numba_pss, numba_cells[0])
print(f"PSS unit time: {pss_time:.2f} ms\n")
 
# 2. concurrency
results = test_concurrency(numba_pss, numba_cells)
 
print(f"PSS concurrency test: {len(numba_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(numba_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

PSS unit time: 3.10 ms

PSS concurrency test: 566 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        3.31     1.00x
       2     2        1.67     1.98x
       4     4        0.86     3.86x
       6     6        0.63     5.25x
      10    10        0.46     7.22x


In [32]:
# 1. unit time
sss_time = time_stage(numba_sss, numba_sss_cells[0])
print(f"Numba SSS unit time: {sss_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(numba_sss, numba_sss_cells)

print(f"Numba SSS concurrency test: {len(numba_sss_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(numba_sss_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

Numba SSS unit time: 0.25 ms

Numba SSS concurrency test: 200 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        0.29     1.00x
       2     2        0.15     1.88x
       4     4        0.10     2.92x
       6     6        0.09     3.11x
      10    10        0.11     2.70x
